# Snowseer

Interactive walkthrough of the cross-season road-overlay pipeline. The
sections mirror [`index.html`](index.html); each runs the relevant
pipeline step on a single curated pair so you can see the system at
work and reproduce any part of it.

Run `make stills` first to populate `data/pairs/` with the curated
manifest pairs. The notebook picks one of them automatically.


## Setup

Resolve the project root, add it to `sys.path`, and pick a curated
pair on disk. The pair the website also uses is preferred; otherwise
fall back to the first available.


In [ ]:
import sys
from pathlib import Path


def _find_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur, *cur.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise RuntimeError(f"Could not find pyproject.toml above {cur}")


ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import cv2
import matplotlib.pyplot as plt
import numpy as np

PAIRS_DIR = ROOT / "data/pairs"
PREFERRED = "gallivare_se__1107706673896225__892247655024233"

candidates = sorted(d for d in PAIRS_DIR.iterdir() if d.is_dir())
preferred_dir = PAIRS_DIR / PREFERRED
if preferred_dir.exists():
    PAIR = preferred_dir
else:
    if not candidates:
        raise SystemExit(
            "No pairs found under data/pairs/. Run `make stills` first."
        )
    PAIR = candidates[0]

print(f"using pair: {PAIR.name}")
snow = cv2.cvtColor(cv2.imread(str(PAIR / "snow.jpg")), cv2.COLOR_BGR2RGB)
clear = cv2.cvtColor(cv2.imread(str(PAIR / "clear.jpg")), cv2.COLOR_BGR2RGB)
print(f"snow shape: {snow.shape}   clear shape: {clear.shape}")


## The constants-bridge

A frozen feature matcher (DISK + LightGlue) finds correspondences
between the snow frame and a paired summer prior of the same
coordinates. The features it can match are the things that don't
change across seasons: gate posts, fence wires, distant roof edges,
masonry corners. No correspondences land on the road surface itself,
because the snow has erased every visual cue there. The homography
fitted to those non-road constants is what carries the road mask
across.


In [ ]:
from src.matching import Matcher, draw_matches
from src.homography import estimate

matcher = Matcher()
match_result = matcher.match(snow, clear)
homo = estimate(match_result, snow.shape[:2], clear.shape[:2])

print(f"matches: {len(match_result.kpts0)}")
print(f"inliers: {homo.n_inliers}")
print(f"ground-plane bias engaged: {homo.used_ground_plane_restriction}")

canvas = draw_matches(
    snow, clear, match_result,
    inlier_mask=homo.inlier_mask,
    max_inliers=25, max_outliers=0,
)
plt.figure(figsize=(14, 6))
plt.imshow(canvas)
plt.axis("off")
plt.title(f"Snow ↔ clear correspondences  (25-inlier subset, {homo.n_inliers} total)")
plt.show()


## The Snowseer system

Same pair, full single-frame pipeline end-to-end:

1. Segment the *clear* prior with a frozen Mask2Former (Cityscapes).
2. Reduce to the largest road component.
3. Warp the road mask through the inverse homography into snow image
   space.
4. Foreground-crop (drop the upper portion the camera should not
   claim as drivable).
5. Alpha-blend onto the snow frame.

Visualise the four constituents in a 2×2 grid: the website renders
the same images via HTML.


In [ ]:
from src.segmentation import RoadSegmenter
from src.overlay import alpha_blend, keep_largest_component, warp_mask
from src.homography import refine_iteratively
from src.fuse import crop_foreground

segmenter = RoadSegmenter()

road_mask_clear = keep_largest_component(segmenter.segment_road(clear))

# Refine the homography if the initial inlier count is low.
if homo.n_inliers < 25:
    refined = refine_iteratively(
        match_result, homo.H, snow.shape[:2], clear.shape[:2], road_mask_clear,
    )
    if refined.H is not None and refined.n_inliers > homo.n_inliers:
        homo = refined
        print(f"refined: inliers {homo.n_inliers}")

H_inv = np.linalg.inv(homo.H)
road_mask_snow = keep_largest_component(warp_mask(road_mask_clear, H_inv, snow.shape[:2]))
overlay_mask = keep_largest_component(crop_foreground(road_mask_snow, foreground_y_frac=0.45))

clear_with_mask = alpha_blend(clear, road_mask_clear, color=(46, 156, 86), alpha=0.50)
overlay = alpha_blend(snow, overlay_mask, color=(46, 156, 86), alpha=0.50)
naive_mask = segmenter.segment_road(snow)
naive = alpha_blend(snow, naive_mask, color=(220, 60, 50), alpha=0.55)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].imshow(snow); axes[0, 0].set_title("Snow input"); axes[0, 0].axis("off")
axes[0, 1].imshow(naive); axes[0, 1].set_title("Naive Cityscapes baseline (red)"); axes[0, 1].axis("off")
axes[1, 0].imshow(clear_with_mask); axes[1, 0].set_title("Clear prior + road mask (green)"); axes[1, 0].axis("off")
axes[1, 1].imshow(overlay); axes[1, 1].set_title("Cross-season overlay (green)"); axes[1, 1].axis("off")
plt.tight_layout()
plt.show()


## Failure mode

The naive baseline (top-right above) is what happens when a Cityscapes-
trained segmenter is applied directly to the snow frame. The road
class spreads across snow, sky, and walls because nothing in the
training distribution looks like a snow-buried street. This is the
failure mode the cross-season pipeline is built to avoid: instead of
guessing what the road looks like under snow, we transfer a known
road mask from a clear-season prior of the same place.


In [ ]:
# Side-by-side: snow input vs naive direct segmentation.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].imshow(snow); axes[0].set_title("Snow input"); axes[0].axis("off")
axes[1].imshow(naive); axes[1].set_title("Cityscapes segmenter, applied directly"); axes[1].axis("off")
plt.tight_layout()
plt.show()

print(f"naive 'road' mask covers {100 * naive_mask.mean():.1f}% of the frame")
print(f"cross-season overlay mask covers {100 * overlay_mask.mean():.1f}%")


## Reproduce

```
git clone https://github.com/aturner22/snowseer; cd snowseer
uv sync --python 3.12
export MAPILLARY_TOKEN=<token>
make reproduce
```

`make track TRACK=<id>` runs a single video track ad-hoc.
`make stills` runs only the static-stills pipeline (the data this
notebook walks through). `make notebook` re-executes this notebook
in place.
